In [ ]:
# %% [0] SETUP GERAL (helpers + config):
# Importando Bibliotecas:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# Pasta onde estão os .parquet exportados
DFS_DIR = globals().get("DFS_DIR", Path("dfs"))

# Limites para não explodir a tela/memória
HEAD_N = 30
TOP_N = 10
SAMPLE_KEYS_N = 5
EVENTS_SHOW_N = 300

def _read_parquet_safe(path: Path, columns=None) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    try:
        return pd.read_parquet(path, columns=columns)
    except TypeError:
        # engines antigos podem não suportar columns
        return pd.read_parquet(path)

def _ensure_day_col(df: pd.DataFrame) -> pd.DataFrame:
    # normaliza para ter _dia_operacional datetime
    if "_dia_operacional" in df.columns:
        df["_dia_operacional"] = pd.to_datetime(df["_dia_operacional"], errors="coerce")
        return df
    if "dia_operacional" in df.columns:
        df["_dia_operacional"] = pd.to_datetime(df["dia_operacional"], errors="coerce")
        return df
    return df

def _ensure_cadastro(df: pd.DataFrame) -> pd.DataFrame:
    if "cadastro" in df.columns:
        df["cadastro"] = df["cadastro"].astype(str).str.strip()
    return df

def load_irreg(filename: str, columns=None) -> pd.DataFrame:
    path = DFS_DIR / filename
    df = _read_parquet_safe(path, columns=columns)
    df = _ensure_cadastro(_ensure_day_col(df))
    return df

def basic_report(df: pd.DataFrame, titulo: str):
    print(f"\n=== {titulo} ===")
    print(f"Linhas: {len(df):,}".replace(",", "."))
    if "cadastro" in df.columns:
        print(f"Cadastros distintos: {df['cadastro'].nunique():,}".replace(",", "."))
    if "_dia_operacional" in df.columns:
        print(f"Dias distintos: {df['_dia_operacional'].nunique():,}".replace(",", "."))
    if ("cadastro" in df.columns) and ("_dia_operacional" in df.columns):
        keys = df[["cadastro", "_dia_operacional"]].dropna().drop_duplicates()
        print(f"Keys (cadastro,dia) distintos: {len(keys):,}".replace(",", "."))
        print("Top 10 cadastros (por nº de dias):")
        display(keys["cadastro"].value_counts().head(TOP_N))
    print("\nHEAD:")
    display(df.head(HEAD_N))

def sample_keys_from_df(df: pd.DataFrame, n=SAMPLE_KEYS_N) -> pd.DataFrame:
    if ("cadastro" not in df.columns) or ("_dia_operacional" not in df.columns):
        return df.head(0)
    keys = df.loc[:, ["cadastro", "_dia_operacional"]].dropna().drop_duplicates()
    return keys.head(n)

def pick_base_events_df():
    # preferir df_valid se existir; senão df
    dfv = globals().get("df_valid")
    if isinstance(dfv, pd.DataFrame):
        return dfv
    dff = globals().get("df")
    if isinstance(dff, pd.DataFrame):
        return dff
    return None

def show_eventos_amostra(keys: pd.DataFrame, titulo: str, cols_wanted=None, extra_filter=None):
    base = pick_base_events_df()
    if base is None:
        print("\n[EVENTOS] df_valid/df não encontrados — vou mostrar apenas as linhas do parquet.")
        return

    if cols_wanted is None:
        cols_wanted = ["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta",
                       "descricao coletor", "descricao_coletor", "desc_coletor", "coletor", "descricao"]

    cols = [c for c in (["cadastro", "_dia_operacional"] + cols_wanted) if c in base.columns]
    ks = keys.copy()
    ks["cadastro"] = ks["cadastro"].astype(str).str.strip()
    ks["_dia_operacional"] = pd.to_datetime(ks["_dia_operacional"], errors="coerce")

    ev = base.loc[:, cols].merge(ks, on=["cadastro", "_dia_operacional"], how="inner")
    if extra_filter is not None:
        ev = ev.loc[extra_filter(ev)]
    if "_dt_ronda" in ev.columns:
        ev["_dt_ronda"] = pd.to_datetime(ev["_dt_ronda"], errors="coerce")
        ev = ev.sort_values(["cadastro", "_dia_operacional", "_dt_ronda"], kind="mergesort")

    print(f"\n[EVENTOS] {titulo} — amostra keys: {len(ks)} | linhas: {len(ev)}")
    display(ev.head(EVENTS_SHOW_N))

def get_desc_col(df: pd.DataFrame) -> str | None:
    for c in ["descricao coletor", "descricao_coletor", "desc_coletor", "coletor", "descricao"]:
        if c in df.columns:
            return c
    return None

def possible_third_parties_ponto_sem_portaria(df_ir_ponto: pd.DataFrame, minutes: int = 2):
    """
    Para amostra de keys do parquet:
    procura no df_valid/df outras pessoas com:
      - mesma planta
      - descricao coletor idêntica (string strip)
      - mesmo dia operacional
      - |diferença de horário| <= minutes
    """
    base = pick_base_events_df()
    if base is None:
        print("\n[TERCEIROS] df_valid/df não encontrados — não dá para cruzar.")
        return

    if not all(c in df_ir_ponto.columns for c in ["cadastro", "_dia_operacional"]):
        print("\n[TERCEIROS] parquet não tem cadastro/_dia_operacional.")
        return

    desc_col = get_desc_col(base)
    if desc_col is None or "planta" not in base.columns or "_dt_ronda" not in base.columns:
        print("\n[TERCEIROS] base não tem colunas necessárias (planta/_dt_ronda/descricao coletor).")
        return

    ks = sample_keys_from_df(df_ir_ponto, SAMPLE_KEYS_N)
    if ks.empty:
        print("\n[TERCEIROS] sem keys.")
        return

    base_use = base.loc[:, [c for c in ["cadastro", "_dia_operacional", "_dt_ronda", "planta", desc_col] if c in base.columns]].copy(deep=False)
    base_use["cadastro"] = base_use["cadastro"].astype(str).str.strip()
    base_use["_dia_operacional"] = pd.to_datetime(base_use["_dia_operacional"], errors="coerce")
    base_use["_dt_ronda"] = pd.to_datetime(base_use["_dt_ronda"], errors="coerce")
    base_use["_desc_norm"] = base_use[desc_col].fillna("").astype(str).str.strip()
    base_use["_planta_num"] = pd.to_numeric(base_use["planta"], errors="coerce")
    base_use = base_use.loc[base_use["_dt_ronda"].notna() & base_use["_desc_norm"].ne("") & base_use["_planta_num"].notna()]

    win_sec = int(minutes * 60)

    print("\n[TERCEIROS] Possíveis pessoas batendo o ponto (mesma planta + desc idêntica + <=2min) — amostra:")
    for _, r in ks.iterrows():
        cad0 = str(r["cadastro"]).strip()
        dia0 = pd.to_datetime(r["_dia_operacional"], errors="coerce")
        if pd.isna(dia0):
            continue

        ev0 = base_use.loc[(base_use["cadastro"].eq(cad0)) & (base_use["_dia_operacional"].eq(dia0))]
        if ev0.empty:
            print(f"- {cad0} / {dia0.date()}: sem eventos base nesse dia.")
            continue

        # limita para não explodir
        ev0 = ev0.head(200)

        counts = {}
        for _, e in ev0.iterrows():
            pl = e["_planta_num"]
            descv = e["_desc_norm"]
            t0 = e["_dt_ronda"]

            m = (
                base_use["_dia_operacional"].eq(dia0)
                & base_use["_planta_num"].eq(pl)
                & base_use["_desc_norm"].eq(descv)
            )
            cand = base_use.loc[m, ["cadastro", "_dt_ronda"]]
            if cand.empty:
                continue

            dif = (cand["_dt_ronda"] - t0).abs()
            hit = cand.loc[dif <= pd.Timedelta(seconds=win_sec), "cadastro"].astype(str).str.strip()
            for c in hit:
                if c == cad0:
                    continue
                counts[c] = counts.get(c, 0) + 1

        print(f"- {cad0} / {dia0.date()}:")
        if not counts:
            print("  nenhum terceiro encontrado.")
        else:
            top = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[:10]
            display(pd.DataFrame(top, columns=["cadastro_possivel", "qtde_matches"]))

In [ ]:
# %% [1] PARQUET — irregularidade_registro_sem_portaria.parquet

df_registro_sem_portaria = load_irreg("irregularidade_registro_sem_portaria.parquet")
basic_report(df_registro_sem_portaria, "Registro sem Portaria — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_registro_sem_portaria)
show_eventos_amostra(
    keys,
    "Registro sem Portaria — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [2] PARQUET — irregularidade_ponto_sem_portaria.parquet (+ terceiros)

df_ponto_sem_portaria = load_irreg("irregularidade_ponto_sem_portaria.parquet")
basic_report(df_ponto_sem_portaria, "Ponto sem Portaria — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_ponto_sem_portaria)
show_eventos_amostra(
    keys,
    "Ponto sem Portaria — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

# Possíveis terceiros batendo o ponto (mesma planta + desc idêntica + <=2min)
possible_third_parties_ponto_sem_portaria(df_ponto_sem_portaria, minutes=2)

In [ ]:
# %% [3] PARQUET — irregularidade_pessoas_trabalhando_sem_ganhar.parquet

df_sem_ganhar = load_irreg("irregularidade_pessoas_trabalhando_sem_ganhar.parquet")
basic_report(df_sem_ganhar, "Trabalhando sem Ganhar — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_sem_ganhar)
show_eventos_amostra(
    keys,
    "Trabalhando sem Ganhar — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [4] PARQUET — irregularidade_pessoas_trabalhando_alem_da_jornada_de_trabalho.parquet

df_alem_jornada = load_irreg("irregularidade_pessoas_trabalhando_alem_da_jornada_de_trabalho.parquet")
basic_report(df_alem_jornada, "Trabalhando Além da Jornada — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_alem_jornada)
show_eventos_amostra(
    keys,
    "Trabalhando Além da Jornada — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [5] PARQUET — irregularidade_ausencia_de_descanso_entre_jornadas_de_trabalho.parquet

df_descanso = load_irreg("irregularidade_ausencia_de_descanso_entre_jornadas_de_trabalho.parquet")
basic_report(df_descanso, "Ausência de Descanso entre Jornadas — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_descanso)
show_eventos_amostra(
    keys,
    "Ausência de Descanso — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Portaria", "_dir_norm", "Presencial", "Ponto", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [6] PARQUET — irregularidade_999_como_trajeto.parquet

df_999_trajeto = load_irreg("irregularidade_999_como_trajeto.parquet")
basic_report(df_999_trajeto, "999 como Trajeto — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_999_trajeto)
show_eventos_amostra(
    keys,
    "999 como Trajeto — eventos (amostra)",
    cols_wanted=["_dt_ronda", "999", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [7] PARQUET — irregularidade_executivos_com_historico_de_999_como_trajeto.parquet

df_exec_999 = load_irreg("irregularidade_executivos_com_historico_de_999_como_trajeto.parquet")
basic_report(df_exec_999, "Executivos com histórico de 999 como trajeto — linhas consideradas (parquet)")

# opcional: conferir se cargo == 'E' existe no parquet
if "cargo" in df_exec_999.columns:
    print("\nDistribuição de cargo (parquet):")
    display(df_exec_999["cargo"].astype(str).str.strip().str.upper().value_counts().head(10))

keys = sample_keys_from_df(df_exec_999)
show_eventos_amostra(
    keys,
    "Exec 999 pré-promo — eventos (amostra)",
    cols_wanted=["_dt_ronda", "999", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor", "cargo"]
)

In [ ]:
# %% [8] PARQUET — irregularidade_ausencia_das_horas_presenciais_minimas.parquet

df_min_horas = load_irreg("irregularidade_ausencia_das_horas_presenciais_minimas.parquet")
basic_report(df_min_horas, "Ausência das horas presenciais mínimas — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_min_horas)
show_eventos_amostra(
    keys,
    "Min horas presenciais — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [9] PARQUET — irregularidade_pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado.parquet

df_feriado = load_irreg("irregularidade_pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado.parquet")
basic_report(df_feriado, "Semana de feriado prolongado — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_feriado)
show_eventos_amostra(
    keys,
    "Feriado prolongado — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)

In [ ]:
# %% [10] PARQUET — irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill.parquet

df_ref_grill = load_irreg("irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill.parquet")
basic_report(df_ref_grill, "Não entrou no refeitório, mas passou no grill — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_ref_grill)

# Se tiver df_valid/df, mostrar só eventos relacionados a REF/GRILL na amostra
base = pick_base_events_df()
desc_col = get_desc_col(base) if isinstance(base, pd.DataFrame) else None

def _filter_ref_grill(ev: pd.DataFrame):
    if desc_col and desc_col in ev.columns:
        s = ev[desc_col].fillna("").astype(str)
        return s.str.contains("REF|REFEIT|GRILL", case=False, na=False, regex=True)
    # fallback: se não tiver descrição, não filtra
    return pd.Series([True] * len(ev), index=ev.index)

show_eventos_amostra(
    keys,
    "REF vs GRILL — eventos (amostra filtrada por texto)",
    cols_wanted=["_dt_ronda", "Portaria", "_dir_norm", "_REFEITORIO", "_GRILL", "planta", "descricao coletor", "descricao_coletor", "coletor", "descricao"],
    extra_filter=_filter_ref_grill
)

In [ ]:
# %% [11] PARQUET — irregularidade_registros_de_pessoas_desligadas.parquet

df_desligadas = load_irreg("irregularidade_registros_de_pessoas_desligadas.parquet")
basic_report(df_desligadas, "Registros de pessoas desligadas — linhas consideradas (parquet)")

keys = sample_keys_from_df(df_desligadas)
show_eventos_amostra(
    keys,
    "Pessoas desligadas — eventos (amostra)",
    cols_wanted=["_dt_ronda", "Presencial", "Ponto", "Portaria", "_dir_norm", "planta", "descricao coletor", "descricao_coletor", "coletor"]
)